# BUG 02 — La muestra cambia sin aviso al agregar un control

**Unidad 4.c** · Acompaña a `Clase_02_VariablesInstrumentales` · Notas: cap. 2

> **Este cuaderno contiene un error deliberado.** No lo corrijas todavía: ejecútalo,
> observa la salida y sigue las tareas del final. Las soluciones están en
> [`SOLUCIONES.md`](SOLUCIONES.md), que conviene no abrir antes de intentarlo.

Estimamos el efecto de la calidad institucional sobre el PIB per cápita con los datos de
Acemoglu, Johnson y Robinson (2001), primero sin controles y luego agregando uno.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

datos = pd.read_stata("../../Clase_02_VariablesInstrumentales/maketable4.dta")

# logpgp95 : log del PIB per cápita en 1995
# avexpr   : protección promedio contra la expropiación (calidad institucional)
# loghjypl : log del producto por trabajador
datos[["shortnam", "logpgp95", "avexpr", "loghjypl"]].head()

,shortnam,logpgp95,avexpr,loghjypl
0,AFG,NaN,NaN,NaN
1,AGO,7.770645,5.363636,-3.411248
2,ARE,9.804219,7.181818,NaN
3,ARG,9.133459,6.386364,-0.872274
4,ARM,7.682482,NaN,NaN


In [2]:
sin_control = smf.ols("logpgp95 ~ avexpr", data=datos).fit()
con_control = smf.ols("logpgp95 ~ avexpr + loghjypl", data=datos).fit()

print("Modelo 1: sin controles")
print(f"  beta_avexpr = {sin_control.params['avexpr']:.4f}")
print(f"  ee          = {sin_control.bse['avexpr']:.4f}")
print(f"  N           = {int(sin_control.nobs)}")

print("\nModelo 2: con el control loghjypl")
print(f"  beta_avexpr = {con_control.params['avexpr']:.4f}")
print(f"  ee          = {con_control.bse['avexpr']:.4f}")
print(f"  N           = {int(con_control.nobs)}")

print(f"\nCambio: {con_control.params['avexpr'] - sin_control.params['avexpr']:+.4f}")

Modelo 1: sin controles
  beta_avexpr = 0.5319
  ee          = 0.0406
  N           = 111

Modelo 2: con el control loghjypl
  beta_avexpr = 0.1856
  ee          = 0.0341
  N           = 102

Cambio: -0.3463


## La conclusión tentadora

> «Al controlar por el producto por trabajador, el efecto de las instituciones se reduce
> a un tercio.»

Antes de creerla, mira los tamaños de muestra de las dos regresiones.

In [3]:
print(f"modelo 1: N = {int(sin_control.nobs)}")
print(f"modelo 2: N = {int(con_control.nobs)}")
print(f"diferencia: {int(sin_control.nobs) - int(con_control.nobs)} países")

print("\nDatos faltantes por variable:")
for variable, cuantos in datos[["logpgp95", "avexpr", "loghjypl"]].isna().sum().items():
    print(f"  {variable:10s} {cuantos:3d} de {len(datos)}")

modelo 1: N = 111
modelo 2: N = 102
diferencia: 9 países

Datos faltantes por variable:
  logpgp95    15 de 163
  avexpr      42 de 163
  loghjypl    40 de 163


`statsmodels` elimina los renglones con faltantes en cualquiera de las variables del
modelo (`missing='drop'` por omisión) y **no lo anuncia**. Las dos regresiones describen
conjuntos de países distintos: comparar sus coeficientes mezcla el efecto de condicionar
con el efecto de cambiar la población.

La comparación, tal como está, no es válida.

## Tareas

1. Pídele a un asistente de IA un procedimiento para separar cuánto del cambio en el
   coeficiente viene del control y cuánto del cambio de muestra.
2. Impleméntalo abajo. La comparación honesta exige estimar **ambos** modelos sobre la
   muestra común.
3. **Antes de ejecutarlo, anota qué esperas encontrar.** El resultado de este caso es
   instructivo justamente porque no es el que se anticipa.
4. A la luz del resultado, ¿la conclusión tentadora queda justificada, refutada, o
   ninguna de las dos?

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las otras actividades de depuración.

In [4]:
# Tu diagnóstico aquí.
#
# Pista: necesitas tres regresiones, no dos.
#   A. sin control, muestra grande
#   B. sin control, muestra común
#   C. con control, muestra común
# El paso A -> B aísla el efecto del cambio de muestra;
# el paso B -> C aísla el efecto del control.